# Persian Emotional TTS
## نوت‌بوک مرحله‌ای پروژه در Google Colab

این نوت‌بوک برای تولید گفتار فارسی با کنترل لحن و احساس، بر اساس نقشه راه پنج‌فازی پروژه تنظیم شده است.

- **محیط اجرا:** Google Colab
- **ذخیره‌سازی پایدار:** Google Drive
- **مسیر پروژه:** `/content/drive/MyDrive/Persian_Emotional_TTS`
- **وضعیت فعلی:** گام ۱.۱ طبق اعلام کاربر انجام شده و در این نوت‌بوک قابل تکرار است؛ گام ۱.۲ آماده اجراست.

> سلول‌ها را به ترتیب و فعلاً فقط تا پایان گام ۱.۲ اجرا کن. سایر گام‌ها فعلاً فقط ساختار و محل مشخص دارند و انجام‌شده محسوب نمی‌شوند.

## نقشه راه پروژه

### فاز ۱: زیرساخت، ابزارها و آماده‌سازی داده
- [x] **۱.۱:** راه‌اندازی Colab، اتصال Drive و ساختار پوشه‌بندی
- [ ] **۱.۲:** نصب کتابخانه‌های کلیدی AI، پردازش صوت و مدل‌سازی
- [ ] **۱.۳:** جمع‌آوری پیکره داده‌های داستانی و صوتی
- [ ] **۱.۴:** پیش‌پردازش متن فارسی (G2P) و Force Alignment
- [ ] **۱.۵:** برچسب‌گذاری احساسی اولیه (Pseudo-Labeling)

### فاز ۲: ماژول درک متن و تحلیل احساس
- [ ] **۲.۱:** انتخاب مدل زبانی سبک فارسی
- [ ] **۲.۲:** استخراج بردارهای Emotion/Prosody از متن

### فاز ۳: مدل آکوستیک و ووکودر
- [ ] **۳.۱:** پیاده‌سازی Flow Matching مشابه Matcha-TTS
- [ ] **۳.۲:** آموزش مدل آکوستیک با شرطی‌سازی احساس
- [ ] **۳.۳:** اتصال BigVGAN برای تولید صوت

### فاز ۴: اجرای بلادرنگ و استریم
- [ ] **۴.۱:** Sentence/Clause Chunking
- [ ] **۴.۲:** Async Pipeline برای کاهش تأخیر
- [ ] **۴.۳:** ONNX و WebSocket Audio Streaming

### فاز ۵: ارزیابی و بهبود
- [ ] **۵.۱:** ارزیابی انسانی و MOS
- [ ] **۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال

## قبل از اجرا

1. نوت‌بوک را در [Google Colab](https://colab.research.google.com/) باز کن.
2. از مسیر **Runtime → Change runtime type → Hardware accelerator** گزینه **GPU** را انتخاب کن.
3. سلول‌ها را از بالا به پایین اجرا کن.
4. هنگام اتصال Google Drive، دسترسی لازم را تأیید کن.
5. پس از اتمام گام ۱.۲، خروجی آخرین سلول را برای بررسی و ورود به گام ۱.۳ نگه دار.

# فاز ۱ — زیرساخت، ابزارها و آماده‌سازی داده
## گام ۱.۱ — بررسی محیط، اتصال Drive و ساختار پروژه

### سلول ۱: بررسی Python، PyTorch و GPU

این سلول نصب جدیدی انجام نمی‌دهد و فقط وضعیت runtime فعلی Colab را گزارش می‌کند.

In [ ]:
import platform
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA runtime:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: Select a GPU from Runtime > Change runtime type.")
except ImportError:
    print("PyTorch is not installed; it will be handled in step 1.2.")

### سلول ۲: اتصال Google Drive

در اولین اجرا، Colab پنجره اجازه دسترسی به Drive را نمایش می‌دهد. حساب موردنظر را انتخاب و دسترسی را تأیید کن.

In [ ]:
from google.colab import drive
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

assert Path("/content/drive/MyDrive").is_dir(), "Google Drive mount failed."
print("Google Drive is ready.")

### سلول ۳: ساخت پوشه‌ها و فایل پیکربندی

اجرای این سلول امن و قابل تکرار است؛ فایل‌های صوتی یا مدل‌های موجود حذف نمی‌شوند.

In [ ]:
import json
from pathlib import Path

PROJECT_NAME = "Persian_Emotional_TTS"
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
DIRECTORIES = (
    "data/raw",
    "data/processed",
    "models/acoustic",
    "models/emotion_extractor",
    "models/vocoder",
    "configs",
)

for relative_path in DIRECTORIES:
    (PROJECT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)
    print(f"Ready: {relative_path}/")

project_config = {
    "project_name": PROJECT_NAME,
    "project_root": str(PROJECT_ROOT),
    "directories": list(DIRECTORIES),
}
config_path = PROJECT_ROOT / "configs" / "project_config.json"
config_path.write_text(
    json.dumps(project_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


### سلول ۴: اعتبارسنجی ساختار پروژه

این سلول وجود همه پوشه‌ها و فایل پیکربندی را بررسی می‌کند.

In [ ]:
missing_directories = [
    relative_path
    for relative_path in DIRECTORIES
    if not (PROJECT_ROOT / relative_path).is_dir()
]
assert not missing_directories, f"Missing directories: {missing_directories}"
assert config_path.is_file(), "Project configuration was not created."

print("Project root:", PROJECT_ROOT)
print("Structure verification: PASSED")
print("Step 1.1 setup: VERIFIED")

## گام ۱.۲ — نصب کتابخانه‌های کلیدی

### سلول ۵: نصب یا تکمیل کتابخانه‌ها

Colab معمولاً PyTorch متناسب با CUDA خودش را از قبل دارد؛ بنابراین نسخه CUDA آن را بی‌دلیل تغییر نمی‌دهیم. اگر PyTorch/Torchaudio/Torchvision موجود نباشند، این سلول آن‌ها را هم نصب می‌کند.

کتابخانه‌های اختصاصی Matcha-TTS و BigVGAN عمداً در این مرحله نصب نمی‌شوند تا در فاز سوم، نسخه‌های سازگار با معماری پروژه انتخاب شوند.

In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
]

for package in ("torch", "torchaudio", "torchvision"):
    if importlib.util.find_spec(package) is None:
        packages.append(package)

print("Installing/checking:", ", ".join(packages))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages],
    check=True,
)
print("Package installation completed.")

### سلول ۶: تست import و نسخه‌های کتابخانه‌ها

اجرای این سلول تایید می‌کند که همه کتابخانه‌های موردنیاز گام ۱.۲ بدون خطا import می‌شوند و در صورت وجود GPU، یک تست ساده CUDA نیز انجام می‌شود.

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version

packages_to_verify = (
    "torch",
    "torchaudio",
    "torchvision",
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
)

failed_imports = []
for package in packages_to_verify:
    try:
        importlib.import_module(package)
        try:
            package_version = version(package)
        except PackageNotFoundError:
            package_version = "version unavailable"
        print(f"OK  {package:<14} {package_version}")
    except Exception as exc:
        failed_imports.append(package)
        print(f"FAIL {package}: {type(exc).__name__}: {exc}")

if failed_imports:
    raise RuntimeError(
        "Step 1.2 is not complete. Failed imports: " + ", ".join(failed_imports)
    )

import torch

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    test_tensor = torch.rand(1024, device="cuda")
    print("CUDA tensor test: PASSED; mean =", round(test_tensor.mean().item(), 4))
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Imports passed, but no GPU is active in this runtime.")

print("\nEnvironment verification: PASSED")

## توقف و گزارش گام ۱.۲

در پایان اجرای موفق باید `Environment verification: PASSED` را ببینی.

فایل `project_config.json` در پوشه `configs` داخل Drive ذخیره شده است. خروجی آخرین سلول را برای بررسی و ورود به گام ۱.۳ ارسال کن.

## گام‌های بعدی فاز ۱ — هنوز پیاده‌سازی نشده‌اند

### گام ۱.۳: جمع‌آوری پیکره داده‌های داستانی و صوتی
در مرحله بعد: تعریف قالب metadata، کنترل حق استفاده از داده، بررسی فرمت و کیفیت نمونه‌ها.

### گام ۱.۴: پیش‌پردازش متن و همگام‌سازی
زیربخش‌ها: نرمال‌سازی متن فارسی، G2P، Force Alignment و ساخت فایل‌های alignment.

### گام ۱.۵: Pseudo-Labeling احساسی
زیربخش‌ها: انتخاب دسته‌های احساس، تولید برچسب اولیه و کنترل کیفیت برچسب‌ها.

# فاز ۲ — Context & Emotion Extractor
## وضعیت: در انتظار تکمیل فاز ۱

- **گام ۲.۱:** انتخاب مدل زبانی سبک فارسی، مانند ParsBERT یا نسخه کوچک Qwen.
- **گام ۲.۲:** Fine-tuning برای استخراج Emotion/Prosody Vectors از متن داستان.

# فاز ۳ — Acoustic Model & Vocoder
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۳.۱:** پیاده‌سازی Flow Matching مشابه Matcha-TTS.
- **گام ۳.۲:** آموزش مدل آکوستیک با شرطی‌سازی بردار احساس.
- **گام ۳.۳:** اتصال BigVGAN برای تولید صوت باکیفیت و حفظ جزئیات عاطفی.

# فاز ۴ — Real-Time Pipeline
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۴.۱:** Sentence/Clause Chunking هوشمند.
- **گام ۴.۲:** طراحی Async Pipeline برای کاهش تأخیر اولیه.
- **گام ۴.۳:** صادرات ONNX و استریم صوت با WebSocket.

# فاز ۵ — Evaluation & Fine-Tuning
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۵.۱:** ارزیابی ذهنی و انسانی کیفیت لحن‌ها با MOS.
- **گام ۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال‌های احتمالی.